#VGG16

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold

BASE_DIR = "/kaggle/input/datamri"
train_dir = os.path.join(BASE_DIR, "Training")
test_dir = os.path.join(BASE_DIR, "Testing")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Load raw datasets
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

# Optimize datasets for training
train_ds = train_ds_raw.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds_raw.cache().prefetch(buffer_size=AUTOTUNE)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


# Factory function for VGG16
def create_vgg16_model():
    base_model = tf.keras.applications.VGG16(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet"
    )
    base_model.trainable = False  # Freeze base model

    model = tf.keras.Sequential([
        tf.keras.layers.Lambda(
            tf.keras.applications.vgg16.preprocess_input
        ),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Parallel Training strategy
strategy = tf.distribute.MirroredStrategy()

# K-Fold Cross-Validation setup
kfold = KFold(n_splits=5, shuffle=True)
fold_no = 1

for train_index, val_index in kfold.split(np.arange(len(train_ds))):
    print(f"\n--- Training Fold {fold_no} ---")

    train_subset = train_ds.take(len(train_index))
    val_subset = train_ds.skip(len(train_index)).take(len(val_index))

    with strategy.scope():
        model = create_vgg16_model()
        model.fit(
            train_subset,
            validation_data=val_subset,
            epochs=2,
            batch_size=BATCH_SIZE,
        )

    fold_no += 1

# Train final model on full training set
print("\n--- Training Final Model on Full Dataset ---")
with strategy.scope():
    final_model = create_vgg16_model()
    final_model.fit(train_ds, epochs=2, batch_size=BATCH_SIZE)

# Evaluate on test set & compute metrics
print("\n--- Evaluating Test Set Performance ---")
y_true = []
y_pred = []

for images, labels in test_ds:
    # Convert one-hot encoded ground truth to class indices
    y_true.extend(np.argmax(labels.numpy(), axis=1))

    # Predict probabilities and get highest probability class
    preds = final_model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"\nFinal Test Accuracy : {acc:.4f}")
print(f"Final Test Precision: {precision:.4f}")
print(f"Final Test Recall   : {recall:.4f}")
print(f"Final Test F1-Score : {f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

ResNet50

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold

BASE_DIR = "/kaggle/input/datamri"
train_dir = os.path.join(BASE_DIR, "Training")
test_dir = os.path.join(BASE_DIR, "Testing")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Load raw datasets
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

# Optimize datasets for training
train_ds = train_ds_raw.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds_raw.cache().prefetch(buffer_size=AUTOTUNE)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


# Factory function for ResNet50
def create_resnet50_model():
    base_model = tf.keras.applications.ResNet50(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet"
    )
    base_model.trainable = False  # Freeze base model

    model = tf.keras.Sequential([
        tf.keras.layers.Lambda(
            tf.keras.applications.resnet50.preprocess_input
        ),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Parallel Training strategy
strategy = tf.distribute.MirroredStrategy()

# K-Fold Cross-Validation setup
kfold = KFold(n_splits=5, shuffle=True)
fold_no = 1

for train_index, val_index in kfold.split(np.arange(len(train_ds))):
    print(f"\n--- Training Fold {fold_no} ---")

    train_subset = train_ds.take(len(train_index))
    val_subset = train_ds.skip(len(train_index)).take(len(val_index))

    with strategy.scope():
        model = create_resnet50_model()
        model.fit(
            train_subset,
            validation_data=val_subset,
            epochs=2,
            batch_size=BATCH_SIZE,
        )

    fold_no += 1

# Train final model on full training set
print("\n--- Training Final Model on Full Dataset ---")
with strategy.scope():
    final_model = create_resnet50_model()
    final_model.fit(train_ds, epochs=2, batch_size=BATCH_SIZE)

# Evaluate on test set & compute metrics
print("\n--- Evaluating Test Set Performance ---")
y_true = []
y_pred = []

for images, labels in test_ds:
    # Convert one-hot encoded ground truth to class indices
    y_true.extend(np.argmax(labels.numpy(), axis=1))

    # Predict probabilities and get highest probability class
    preds = final_model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"\nFinal Test Accuracy : {acc:.4f}")
print(f"Final Test Precision: {precision:.4f}")
print(f"Final Test Recall   : {recall:.4f}")
print(f"Final Test F1-Score : {f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

Inception ResNet V2

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold

BASE_DIR = "/kaggle/input/datamri"
train_dir = os.path.join(BASE_DIR, "Training")
test_dir = os.path.join(BASE_DIR, "Testing")

IMG_SIZE = (299, 299)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Load raw datasets
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

# Optimize datasets for training
train_ds = train_ds_raw.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds_raw.cache().prefetch(buffer_size=AUTOTUNE)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


# Factory function for InceptionResNetV2
def create_inception_resnet_v2_model():
    base_model = tf.keras.applications.InceptionResNetV2(
        input_shape=(299, 299, 3), include_top=False, weights="imagenet"
    )
    base_model.trainable = False  # Freeze base model

    model = tf.keras.Sequential([
        tf.keras.layers.Lambda(
            tf.keras.applications.inception_resnet_v2.preprocess_input
        ),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Parallel Training strategy
strategy = tf.distribute.MirroredStrategy()

# K-Fold Cross-Validation setup
kfold = KFold(n_splits=5, shuffle=True)
fold_no = 1

for train_index, val_index in kfold.split(np.arange(len(train_ds))):
    print(f"\n--- Training Fold {fold_no} ---")

    train_subset = train_ds.take(len(train_index))
    val_subset = train_ds.skip(len(train_index)).take(len(val_index))

    with strategy.scope():
        model = create_inception_resnet_v2_model()
        model.fit(
            train_subset,
            validation_data=val_subset,
            epochs=2,
            batch_size=BATCH_SIZE,
        )

    fold_no += 1

# Train final model on full training set
print("\n--- Training Final Model on Full Dataset ---")
with strategy.scope():
    final_model = create_inception_resnet_v2_model()
    final_model.fit(train_ds, epochs=2, batch_size=BATCH_SIZE)

# Evaluate on test set & compute metrics
print("\n--- Evaluating Test Set Performance ---")
y_true = []
y_pred = []

for images, labels in test_ds:
    # Convert one-hot encoded ground truth to class indices
    y_true.extend(np.argmax(labels.numpy(), axis=1))

    # Predict probabilities and get highest probability class
    preds = final_model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"\nFinal Test Accuracy : {acc:.4f}")
print(f"Final Test Precision: {precision:.4f}")
print(f"Final Test Recall   : {recall:.4f}")
print(f"Final Test F1-Score : {f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

InceptionV3

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold

BASE_DIR = "/kaggle/input/datamri"
train_dir = os.path.join(BASE_DIR, "Training")
test_dir = os.path.join(BASE_DIR, "Testing")

IMG_SIZE = (299, 299)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Load raw datasets
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

# Optimize datasets for training
train_ds = train_ds_raw.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds_raw.cache().prefetch(buffer_size=AUTOTUNE)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


# Factory function for InceptionV3
def create_inceptionv3_model():
    base_model = tf.keras.applications.InceptionV3(
        input_shape=(299, 299, 3), include_top=False, weights="imagenet"
    )
    base_model.trainable = False  # Freeze base model

    model = tf.keras.Sequential([
        tf.keras.layers.Lambda(
            tf.keras.applications.inception_v3.preprocess_input
        ),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Parallel Training strategy
strategy = tf.distribute.MirroredStrategy()

# K-Fold Cross-Validation setup
kfold = KFold(n_splits=5, shuffle=True)
fold_no = 1

for train_index, val_index in kfold.split(np.arange(len(train_ds))):
    print(f"\n--- Training Fold {fold_no} ---")

    train_subset = train_ds.take(len(train_index))
    val_subset = train_ds.skip(len(train_index)).take(len(val_index))

    with strategy.scope():
        model = create_inceptionv3_model()
        model.fit(
            train_subset,
            validation_data=val_subset,
            epochs=2,
            batch_size=BATCH_SIZE,
        )

    fold_no += 1

# Train final model on full training set
print("\n--- Training Final Model on Full Dataset ---")
with strategy.scope():
    final_model = create_inceptionv3_model()
    final_model.fit(train_ds, epochs=2, batch_size=BATCH_SIZE)

# Evaluate on test set & compute metrics
print("\n--- Evaluating Test Set Performance ---")
y_true = []
y_pred = []

for images, labels in test_ds:
    # Convert one-hot encoded ground truth to class indices
    y_true.extend(np.argmax(labels.numpy(), axis=1))

    # Predict probabilities and get highest probability class
    preds = final_model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"\nFinal Test Accuracy : {acc:.4f}")
print(f"Final Test Precision: {precision:.4f}")
print(f"Final Test Recall   : {recall:.4f}")
print(f"Final Test F1-Score : {f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

AlexNet

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold

BASE_DIR = "/kaggle/input/datamri"
train_dir = os.path.join(BASE_DIR, "Training")
test_dir = os.path.join(BASE_DIR, "Testing")

IMG_SIZE = (227, 227)  # Standard input resolution for AlexNet
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Load raw datasets
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

# Optimize datasets for training
train_ds = train_ds_raw.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds_raw.cache().prefetch(buffer_size=AUTOTUNE)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


# Factory function for AlexNet
def create_alexnet_model():
    model = tf.keras.Sequential([
        # Standard input normalization [0, 1]
        tf.keras.layers.Rescaling(1.0 / 255, input_shape=(227, 227, 3)),
        
        # Conv 1
        tf.keras.layers.Conv2D(96, kernel_size=(11, 11), strides=(4, 4), activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),
        
        # Conv 2
        tf.keras.layers.Conv2D(256, kernel_size=(5, 5), padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),
        
        # Conv 3, 4, 5
        tf.keras.layers.Conv2D(384, kernel_size=(3, 3), padding="same", activation="relu"),
        tf.keras.layers.Conv2D(384, kernel_size=(3, 3), padding="same", activation="relu"),
        tf.keras.layers.Conv2D(256, kernel_size=(3, 3), padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),
        
        # FC Head
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(4096, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(4096, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Parallel Training strategy
strategy = tf.distribute.MirroredStrategy()

# K-Fold Cross-Validation setup
kfold = KFold(n_splits=5, shuffle=True)
fold_no = 1

for train_index, val_index in kfold.split(np.arange(len(train_ds))):
    print(f"\n--- Training Fold {fold_no} ---")

    train_subset = train_ds.take(len(train_index))
    val_subset = train_ds.skip(len(train_index)).take(len(val_index))

    with strategy.scope():
        model = create_alexnet_model()
        model.fit(
            train_subset,
            validation_data=val_subset,
            epochs=2,
            batch_size=BATCH_SIZE,
        )

    fold_no += 1

# Train final model on full training set
print("\n--- Training Final Model on Full Dataset ---")
with strategy.scope():
    final_model = create_alexnet_model()
    final_model.fit(train_ds, epochs=2, batch_size=BATCH_SIZE)

# Evaluate on test set & compute metrics
print("\n--- Evaluating Test Set Performance ---")
y_true = []
y_pred = []

for images, labels in test_ds:
    # Convert one-hot encoded ground truth to class indices
    y_true.extend(np.argmax(labels.numpy(), axis=1))

    # Predict probabilities and get highest probability class
    preds = final_model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"\nFinal Test Accuracy : {acc:.4f}")
print(f"Final Test Precision: {precision:.4f}")
print(f"Final Test Recall   : {recall:.4f}")
print(f"Final Test F1-Score : {f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))